In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# load files
os.chdir('..')

raw_ihs_savings_2015 = pd.read_csv('source/householdsavings_15.csv')
raw_ihs_income_2015 = pd.read_csv('source/householdincome_15.CSV')
raw_finscope_2019 = pd.read_csv('source/finscope_19.csv')
raw_dhs_2020 = pd.read_stata('source/household_19_20.DTA')
raw_findex_2021 = pd.read_csv('source/connectivity_21.csv')
raw_findex_2024 = pd.read_csv('source/connectivity_24.csv')

C:\Users\DELL\AppData\Local\Temp\ipykernel_3828\2547714251.py:6: DtypeWarning: Columns (57,58,173,194,271,329,341,377,381,397,403,480,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,625,633,733,736,741,800,802,806,859,860,861,862,863,864,865,866,867,978,980,1061,1062,1072,1085,1172,1173,1191,1192,1193,1194) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_finscope_2019 = pd.read_csv('source/finscope_19.csv')


In [3]:
# 2015 IHS savings account
from pandas import NA


df_savings_2015 = pd.DataFrame()
df_savings_2015['hh_id'] = raw_ihs_savings_2015['hid']
df_savings_2015['settlement'] = raw_ihs_savings_2015['settlement']

df_savings_2015['hh_savings'] = raw_ihs_savings_2015['s7cq1'].map({1.0: 1, 2.0: 0}).fillna(0).astype(int)
df_savings_2015['rural'] = raw_ihs_savings_2015['area'].map({1.0: 0, 2.0: 1}).fillna(0).astype(int)

df_savings_2015['survey_year'] = 2015
df_savings_2015['data_source'] = 'IHS'
df_savings_2015['weight'] = NA

# clean duplicates
df_savings_2015['info_count'] = df_savings_2015.notna().sum(axis=1)
df_savings_2015_sorted = df_savings_2015.sort_values(
    by=['hh_id', 'info_count'], 
    ascending=[True, False]
)
df_savings_2015_clean = df_savings_2015_sorted.drop_duplicates(
    subset=['hh_id'], 
    keep='first'
).copy()
df_savings_2015 = df_savings_2015_clean.drop(columns=['info_count'])

In [4]:
df_savings_2015

,hh_id,settlement,hh_savings,rural,survey_year,data_source,weight
0,1010101,10101,1,0,2015,IHS,<NA>
2,1010103,10101,1,0,2015,IHS,<NA>
3,1010104,10101,1,0,2015,IHS,<NA>
4,1010105,10101,1,0,2015,IHS,<NA>
5,1010106,10101,1,0,2015,IHS,<NA>
...,...,...,...,...,...,...,...
15806,8622216,86231,0,1,2015,IHS,<NA>
15807,8622217,86231,0,1,2015,IHS,<NA>
15808,8622218,86231,0,1,2015,IHS,<NA>
15809,8622219,86231,0,1,2015,IHS,<NA>


In [5]:
# 2019 finscope savings
df_savings_2019 = pd.DataFrame()

df_savings_2019['hh_id'] = raw_finscope_2019['ID'] 
df_savings_2019['hh_savings'] = raw_finscope_2019['I4A'].map({"1. Yes": 1, "2. No": 0}).fillna(0).astype(int)
df_savings_2019['rural'] = raw_finscope_2019['HHID9'].map({"Rural": 1, "Urban": 0}).fillna(0).astype(int)
df_savings_2019['settlement'] = raw_finscope_2019['SETTLEMENT'].str.extract(r'^(\d+)')


df_savings_2019['survey_year'] = 2019
df_savings_2019['data_source'] = 'Finscope'
df_savings_2019['weight'] = NA

In [6]:
df_savings_2019

,hh_id,hh_savings,rural,settlement,survey_year,data_source,weight
0,1,0,0,10201,2019,Finscope,<NA>
1,26,1,0,10201,2019,Finscope,<NA>
2,55,0,0,10201,2019,Finscope,<NA>
3,77,1,0,10201,2019,Finscope,<NA>
4,102,1,0,10201,2019,Finscope,<NA>
...,...,...,...,...,...,...,...
1465,37066,0,1,86225,2019,Finscope,<NA>
1466,37099,0,1,86225,2019,Finscope,<NA>
1467,37121,0,1,86225,2019,Finscope,<NA>
1468,37140,1,1,86225,2019,Finscope,<NA>


In [7]:
# 2019-2020 DHS bank account
df_savings_2020 = pd.DataFrame()

df_savings_2020['hh_id'] = raw_dhs_2020['hhid']
df_savings_2020['survey_year'] = raw_dhs_2020['hv007']
df_savings_2020['rural'] = raw_dhs_2020['hv025'].map({2: 1, 1: 0}).fillna(0).astype(int)
df_savings_2020['hh_savings'] = raw_dhs_2020['hv247'].map({1: 1, 0:0}).fillna(0).astype(int)

df_savings_2020['data_source'] = 'DHS'
df_savings_2020['weight'] = raw_dhs_2020['hv005'] / 1e6

In [8]:
df_savings_2020.head()

,hh_id,survey_year,rural,hh_savings,data_source,weight
0,1 1,2019,0,0,DHS,0.172113
1,1 6,2019,0,0,DHS,0.172113
2,1 10,2019,0,0,DHS,0.172113
3,1 14,2019,0,0,DHS,0.172113
4,1 18,2019,0,0,DHS,0.172113


In [9]:
# 2021 Findex savings
df_savings_2021 = pd.DataFrame()

df_savings_2021['hh_id'] = raw_findex_2021.index.map(lambda x: f"findex_21_{x}")
df_savings_2021['survey_year'] = 2021
df_savings_2021['data_source'] = 'Findex'

df_savings_2021['hh_savings'] = raw_findex_2021['saved'].map({1: 1, 0: 0}).fillna(0).astype(int)
df_savings_2021['rural'] = raw_findex_2021['urbanicity_f2f'].map({1: 1, 2: 0}).fillna(0).astype(int)
df_savings_2021['weight'] = raw_findex_2021['wgt'].fillna(NA)

In [10]:
df_savings_2021

,hh_id,survey_year,data_source,hh_savings,rural,weight
0,findex_21_0,2021,Findex,1,0,0.496558
1,findex_21_1,2021,Findex,0,0,1.878779
2,findex_21_2,2021,Findex,0,0,0.220206
3,findex_21_3,2021,Findex,0,0,0.939390
4,findex_21_4,2021,Findex,0,0,0.504958
...,...,...,...,...,...,...
995,findex_21_995,2021,Findex,0,0,2.215698
996,findex_21_996,2021,Findex,0,1,1.351114
997,findex_21_997,2021,Findex,1,1,0.496418
998,findex_21_998,2021,Findex,1,0,0.373981


In [11]:
# 2024 Findex savings
df_savings_2024 = pd.DataFrame()

df_savings_2024['hh_id'] = raw_findex_2024.index.map(lambda x: f"findex_24_{x}")
df_savings_2024['survey_year'] = 2024
df_savings_2024['data_source'] = 'Findex'

df_savings_2024['hh_savings'] = raw_findex_2024['saved'].map({1: 1, 0: 0}).fillna(0).astype(int)
df_savings_2024['rural'] = raw_findex_2024['urbanicity'].map({1: 1, 2: 0}).fillna(0).astype(int)
df_savings_2024['weight'] = raw_findex_2024['wgt'].fillna(NA)

In [12]:
df_savings_2024

,hh_id,survey_year,data_source,hh_savings,rural,weight
0,findex_24_0,2024,Findex,1,0,0.344393
1,findex_24_1,2024,Findex,0,1,0.609212
2,findex_24_2,2024,Findex,0,0,0.798520
3,findex_24_3,2024,Findex,0,0,0.253243
4,findex_24_4,2024,Findex,1,0,0.531983
...,...,...,...,...,...,...
1003,findex_24_1003,2024,Findex,0,0,0.328288
1004,findex_24_1004,2024,Findex,1,1,0.777492
1005,findex_24_1005,2024,Findex,1,1,0.671328
1006,findex_24_1006,2024,Findex,1,0,0.483620


In [13]:
# contruct panel for savings
df_savings = pd.concat([
    df_savings_2015,
    df_savings_2019,
    df_savings_2020,
    df_savings_2021,
    df_savings_2024
], axis=0, ignore_index=True)

df_savings.to_csv('output/did_panel_savings.csv', index=False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_3828\3614358247.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_savings = pd.concat([


In [14]:
df_savings

,hh_id,settlement,hh_savings,rural,survey_year,data_source,weight
0,1010101,10101,1,0,2015,IHS,NaN
1,1010103,10101,1,0,2015,IHS,NaN
2,1010104,10101,1,0,2015,IHS,NaN
3,1010105,10101,1,0,2015,IHS,NaN
4,1010106,10101,1,0,2015,IHS,NaN
...,...,...,...,...,...,...,...
23303,findex_24_1003,NaN,0,0,2024,Findex,0.328288
23304,findex_24_1004,NaN,1,1,2024,Findex,0.777492
23305,findex_24_1005,NaN,1,1,2024,Findex,0.671328
23306,findex_24_1006,NaN,1,0,2024,Findex,0.483620
